In [0]:
from pyspark.sql.functions import (
    col,
    to_date,
    year,
    month,
    dayofmonth,
    dayofweek,
    weekofyear,
    quarter,
    date_format
)

realtime_date_dim = (
    silver_realtime_df
    .select(
        to_date(col("market_timestamp")).alias("date")
    )
    .distinct()
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("week_of_year", weekofyear(col("date")))
    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "day",
        "day_of_week",
        "week_of_year"
    )
)

display(realtime_date_dim)

In [0]:
realtime_df.printSchema()

In [0]:
silver_base = "s3://stocks-silver-layer/"

In [0]:
# Transform Finnhub data
from pyspark.sql.functions import (
    col,
    lit,
    from_unixtime,
    current_timestamp
)

silver_realtime_df = (
    realtime_df
    .select(
        lit("AAPL").alias("ticker"),
        col("o").alias("open"),
        col("h").alias("high"),
        col("l").alias("low"),
        col("c").alias("close"),
        col("pc").alias("previous_close"),
        col("d").alias("change"),
        col("dp").alias("change_percent"),
        from_unixtime(col("t")).alias("market_timestamp"),
        current_timestamp().alias("ingested_at")
    )
)

display(silver_realtime_df)

In [0]:
# Save to silver layer
silver_realtime_path = "s3://stocks-silver-layer/processed/realtime/"

silver_realtime_df.write \
    .mode("append") \
    .parquet(silver_realtime_path)

# verify
display(
    spark.read
    .parquet(silver_realtime_path)
    .orderBy(col("market_timestamp").desc())
    .limit(10)
)

GOLD LAYER

In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    to_date,
    year,
    month,
    dayofmonth,
    dayofweek,
    weekofyear,
    quarter,
    date_format,
    avg,
    stddev
)
from pyspark.sql.window import Window

# ============================================================
# PATHS
# ============================================================

GOLD_BASE = "s3://stocks-gold-layer/"

DATE_REALTIME_PATH = GOLD_BASE + "dim_date_realtime/"
FACT_REALTIME_PATH = GOLD_BASE + "fact_stock_prices_realtime/"

# EXISTING SHARED DIMENSION
DIM_STOCK_PATH = GOLD_BASE + "dim_stock/"


# ============================================================
# 1. LOAD EXISTING SHARED DIM_STOCK
# ============================================================

dim_stock = spark.read.parquet(DIM_STOCK_PATH)

print("Existing dim_stock loaded:")
print(f"Rows: {dim_stock.count()}")


# ============================================================
# 2. USE EXISTING REAL-TIME SILVER DATA
# ============================================================

realtime = (
    silver_realtime_df
    .filter(col("ticker").isNotNull())
    .filter(col("open").isNotNull())
    .filter(col("high").isNotNull())
    .filter(col("low").isNotNull())
    .filter(col("close").isNotNull())
)


# ============================================================
# 3. CREATE REAL-TIME DATE DIMENSION
# ============================================================

realtime_date_dim = (
    realtime
    .select(
        to_date(col("market_timestamp")).alias("date")
    )
    .distinct()

    .withColumn(
        "date_key",
        date_format(col("date"), "yyyyMMdd").cast("int")
    )

    .withColumn("year", year(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("week_of_year", weekofyear(col("date")))

    .select(
        "date_key",
        "date",
        "year",
        "quarter",
        "month",
        "day",
        "day_of_week",
        "week_of_year"
    )
)


# ============================================================
# 4. SAVE REAL-TIME DATE DIMENSION
# ============================================================

realtime_date_dim.write \
    .mode("append") \
    .partitionBy("year", "month") \
    .parquet(DATE_REALTIME_PATH)


# ============================================================
# 5. JOIN TO EXISTING SHARED DIM_STOCK
# ============================================================

realtime_fact = (
    realtime

    .withColumn(
        "date_key",
        date_format(
            to_date(col("market_timestamp")),
            "yyyyMMdd"
        ).cast("int")
    )

    .join(
        dim_stock.select(
            "stock_key",
            "ticker"
        ),
        on="ticker",
        how="left"
    )
)


# ============================================================
# 6. CHECK STOCK KEY
# ============================================================

missing_stock_key = realtime_fact.filter(
    col("stock_key").isNull()
).count()

print(f"Records missing stock_key: {missing_stock_key}")

if missing_stock_key > 0:
    display(
        realtime_fact.filter(
            col("stock_key").isNull()
        )
    )
    raise Exception(
        "Real-time record does not exist in dim_stock."
    )


# ============================================================
# 7. CREATE REAL-TIME GOLD FACT
# ============================================================

realtime_fact = (
    realtime_fact

    .select(
        "date_key",
        "stock_key",

        col("open").cast("double").alias("open"),
        col("high").cast("double").alias("high"),
        col("low").cast("double").alias("low"),
        col("close").cast("double").alias("close"),

        # Finnhub does not provide adjusted close
        col("close").cast("double").alias("adj_close"),

        # Finnhub quote does not provide volume
        lit(None).cast("long").alias("volume"),

        col("previous_close")
            .cast("double")
            .alias("previous_close"),

        (
            (col("close") - col("previous_close"))
            / col("previous_close")
        )
        .cast("double")
        .alias("daily_return"),

        (
            col("high") - col("low")
        )
        .cast("double")
        .alias("daily_range"),

        col("market_timestamp"),
        col("ingested_at")
    )
)


# ============================================================
# 8. CALCULATE REAL-TIME MOVING AVERAGES
# ============================================================

window_7 = (
    Window
    .partitionBy("stock_key")
    .orderBy("market_timestamp")
    .rowsBetween(-6, 0)
)

window_30 = (
    Window
    .partitionBy("stock_key")
    .orderBy("market_timestamp")
    .rowsBetween(-29, 0)
)

realtime_fact = (
    realtime_fact

    .withColumn(
        "moving_avg_7d",
        avg("close").over(window_7)
    )

    .withColumn(
        "moving_avg_30d",
        avg("close").over(window_30)
    )

    .withColumn(
        "volatility_30d",
        stddev("daily_return").over(window_30)
    )
)


# ============================================================
# 9. REMOVE DUPLICATES WITHIN THIS BATCH
# ============================================================

realtime_fact = (
    realtime_fact
    .dropDuplicates(
        ["stock_key", "market_timestamp"]
    )
)


# ============================================================
# 10. SAVE REAL-TIME GOLD FACT
# ============================================================

realtime_fact.write \
    .mode("append") \
    .partitionBy("date_key") \
    .parquet(FACT_REALTIME_PATH)


# ============================================================
# 11. VERIFY
# ============================================================

saved_dates = spark.read.parquet(
    DATE_REALTIME_PATH
)

saved_realtime_fact = spark.read.parquet(
    FACT_REALTIME_PATH
)

print("=" * 60)
print("REAL-TIME GOLD PIPELINE COMPLETE")
print("=" * 60)

print(
    "Real-time date records:",
    saved_dates.count()
)

print(
    "Real-time fact records:",
    saved_realtime_fact.count()
)

print("\nREAL-TIME DATE DIMENSION")
display(
    saved_dates
    .orderBy(col("date_key").desc())
    .limit(10)
)

print("\nREAL-TIME GOLD FACT")
display(
    saved_realtime_fact
    .orderBy(col("market_timestamp").desc())
    .limit(10)
)

In [0]:
# from pyspark.sql.functions import (
#     col,
#     to_date,
#     date_format,
#     lit
# )

# realtime_gold = (
#     silver_realtime_df
#     .withColumn("trade_date", to_date(col("market_timestamp")))
#     .withColumn(
#         "date_key",
#         date_format(col("trade_date"), "yyyyMMdd").cast("int")
#     )
# )

In [0]:
# # Get the stock key
# realtime_gold = (
#     realtime_gold
#     .join(
#         dim_stock.select("stock_key", "ticker"),
#         on="ticker",
#         how="left"
#     )
# )

# display(realtime_gold)

In [0]:
# display(
#     realtime_gold.filter(col("stock_key").isNull())
# )

In [0]:
# # Check your existing fact schema
# fact_stock_prices.printSchema()

In [0]:
# print(fact_stock_prices.columns)

In [0]:
# from pyspark.sql.functions import (
#     col,
#     lit,
#     when
# )

# realtime_gold_final = (
#     realtime_gold
#     .select(
#         col("date_key"),
#         col("stock_key"),

#         col("open"),
#         col("high"),
#         col("low"),
#         col("close"),

#         # Finnhub quote does not provide adjusted close
#         col("close").alias("adj_close"),

#         # Finnhub quote does not provide volume
#         lit(None).cast("long").alias("volume"),

#         col("previous_close"),

#         # Daily return
#         when(
#             col("previous_close") != 0,
#             (col("close") - col("previous_close"))
#             / col("previous_close")
#         ).alias("daily_return"),

#         # We only have one real-time observation here.
#         # Moving averages will be calculated after multiple observations exist.
#         lit(None).cast("double").alias("moving_avg_7d"),
#         lit(None).cast("double").alias("moving_avg_30d"),

#         # Intraday high-low range
#         (col("high") - col("low")).alias("daily_range"),

#         # Need sufficient historical observations for 30-day volatility
#         lit(None).cast("double").alias("volatility_30d")
#     )
# )

# display(realtime_gold_final)

In [0]:
# print(realtime_gold_final.columns)

In [0]:
# display(
#     realtime_gold_final
#     .select("date_key", "stock_key", "close", "previous_close")
# )

In [0]:
# # Remove duplicates against existing Gold
# existing_keys = (
#     fact_stock_prices
#     .select("date_key", "stock_key", "close")
#     .dropDuplicates()
# )

# new_gold_records = (
#     realtime_gold_final
#     .join(
#         existing_keys,
#         on=["date_key", "stock_key", "close"],
#         how="left_anti"
#     )
# )

# display(new_gold_records)

In [0]:
# new_gold_records.write \
#     .mode("append") \
#     .parquet(gold_base + "fact_stock_prices_realtime/")

# # verify the insert 
# updated_fact = spark.read.parquet(
#     gold_base + "fact_stock_prices_realtime/"
# )

# print("Previous rows:", fact_stock_prices.count())
# print("Current rows:", updated_fact.count())

In [0]:
# from pyspark.sql.window import Window
# from pyspark.sql.functions import avg, stddev

# price_window_7 = (
#     Window
#     .partitionBy("stock_key")
#     .orderBy("date_key")
#     .rowsBetween(-6, 0)
# )

# price_window_30 = (
#     Window
#     .partitionBy("stock_key")
#     .orderBy("date_key")
#     .rowsBetween(-29, 0)
# )

# gold_with_moving_avg = (
#     updated_fact
#     .withColumn(
#         "moving_avg_7d",
#         avg("close").over(price_window_7)
#     )
#     .withColumn(
#         "moving_avg_30d",
#         avg("close").over(price_window_30)
#     )
#     .withColumn(
#         "volatility_30d",
#         stddev("daily_return").over(price_window_30)
#     )
# )

# display(
#     gold_with_moving_avg
#     .filter(col("stock_key") == realtime_gold_final.first()["stock_key"])
#     .orderBy(col("date_key").desc())
#     .limit(10)
# )

In [0]:
# display(
#     updated_fact
#     .orderBy(col("date_key").desc())
#     .limit(10)
# )

# print("Total Gold rows:", updated_fact.count())

In [0]:
# batch_gold = spark.read.parquet(
#     "s3://stocks-gold-layer/fact_stock_prices/"
# )

# print("Current Gold rows:", batch_gold.count())

In [0]:
# display(
#     batch_gold
#     .join(
#         dim_stock.select("stock_key", "ticker"),
#         "stock_key"
#     )
#     .filter(col("ticker") == "AAPL")
#     .orderBy(col("date_key").desc())
#     .limit(20)
# )

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    DoubleType,
    LongType
)

bronze_path = "s3://stocks-bronze-layer/raw/realtime/"

checkpoint_path = (
    "s3://stocks-bronze-layer/checkpoints/realtime/"
)

schema = StructType([
    StructField("c", DoubleType(), True),
    StructField("d", DoubleType(), True),
    StructField("dp", DoubleType(), True),
    StructField("h", DoubleType(), True),
    StructField("l", DoubleType(), True),
    StructField("o", DoubleType(), True),
    StructField("pc", DoubleType(), True),
    StructField("t", LongType(), True)
])

realtime_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option(
        "cloudFiles.schemaLocation",
        checkpoint_path + "schema/"
    )
    .schema(schema)
    .load(bronze_path)
)

display(
    realtime_stream,
    checkpointLocation=checkpoint_path + "display/"
)

call real time files

In [0]:
spark.readStream.format("cloudFiles")